# División del dataset — train / val / test

Aplica `src/data/make_splits.py` y muestra el resultado:

1. **Quitar duplicados** con los grupos que encontró `eda.ipynb`: de cada grupo con la misma etiqueta se queda
   una imagen (la de mayor resolución); los grupos con etiquetas distintas se quitan completos.
2. **Dividir 70/15/15**, estratificado por clase y con semilla fija.
3. **Verificar** que no haya fugas entre partes y que cada parte se parezca al dataset completo, también en cámaras.
4. **Guardar** `data/splits.csv`, que se sube al repo y usan todos los entrenamientos.

**Cómo correrlo (SageMaker Studio):** primero `eda.ipynb` (deja `duplicates.csv` e `image_stats.csv` en `DATA_DIR`),
luego *Run All* aquí. Tarda segundos.

In [ ]:
import os
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

REPO = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
sys.path.insert(0, str(REPO))

try:
    from dotenv import load_dotenv
    load_dotenv(REPO / ".env")
except ImportError:
    pass

from src.common.preprocess import crop_retina, load_image
from src.data.make_splits import SEED, SPLITS, drop_duplicates, split_dataset

DATA_DIR = Path(os.environ.get("DATA_DIR", REPO / "data" / "aptos")).expanduser()
OUT = REPO / "data" / "splits.csv"
CLASS_NAMES = {0: "Sin retinopatía", 1: "Leve", 2: "Moderada", 3: "Severa", 4: "Proliferativa"}
SPLIT_ORDER = ["train", "val", "test"]
SPLIT_COLORS = {"train": "#2e75b6", "val": "#f5a623", "test": "#2e7d32"}
plt.rcParams["figure.dpi"] = 90

for name in ["train.csv", "duplicates.csv", "image_stats.csv"]:
    assert (DATA_DIR / name).is_file(), f"Falta {DATA_DIR / name}: corre primero eda.ipynb"

labels = pd.read_csv(DATA_DIR / "train.csv")
duplicates = pd.read_csv(DATA_DIR / "duplicates.csv")
stats = pd.read_csv(DATA_DIR / "image_stats.csv")
stats["resolution"] = stats.height.astype(int).astype(str) + "×" + stats.width.astype(int).astype(str)
print(f"DATA_DIR = {DATA_DIR}")
print(f"{len(labels)} imágenes · {duplicates.dup_group.nunique()} grupos de duplicados con {len(duplicates)} imágenes")

## 1. Quitar duplicados

In [ ]:
kept, removed = drop_duplicates(labels, duplicates, stats)

summary = removed.groupby("reason").agg(grupos=("dup_group", "nunique"), imágenes=("id_code", "size"))
display(summary)
print(f"Quedan {len(kept)} de {len(labels)} imágenes ({len(removed)} quitadas)")

Ejemplos: a la izquierda de cada fila la imagen que **se queda** (marco verde), a la derecha las que **se quitan**
(marco rojo). Las dos últimas filas son grupos con etiquetas distintas, que se quitan completos.

In [ ]:
def show_groups(group_ids, max_cols=3):
    fig, axes = plt.subplots(len(group_ids), max_cols, figsize=(3.2 * max_cols, 3.4 * len(group_ids)), squeeze=False)
    for row, g in enumerate(group_ids):
        members = duplicates.loc[duplicates.dup_group == g, ["id_code", "dup_group"]].merge(labels, on="id_code").merge(stats[["id_code", "resolution"]], on="id_code")
        members["kept"] = members.id_code.isin(kept.id_code)
        members = members.sort_values("kept", ascending=False).head(max_cols)
        for col in range(max_cols):
            ax = axes[row, col]
            ax.set_xticks([]); ax.set_yticks([])
            if col >= len(members):
                ax.axis("off"); continue
            r = members.iloc[col]
            ax.imshow(crop_retina(load_image(DATA_DIR / "train_images" / f"{r.id_code}.png")))
            color = "#2e7d32" if r.kept else "#c62828"
            for side in ax.spines.values():
                side.set_edgecolor(color); side.set_linewidth(4)
            ax.set_title(f"{'se queda' if r.kept else 'se quita'} · clase {r.diagnosis}\n{r.resolution}", fontsize=9, color=color)
    plt.tight_layout()
    plt.show()


copies = removed[removed.reason == "copia"].dup_group.unique()[:3]
conflicts = removed[removed.reason == "etiquetas distintas"].dup_group.unique()[:2]
show_groups(list(copies) + list(conflicts))

Cómo cambia la distribución de clases al quitar duplicados:

In [ ]:
before = labels.diagnosis.value_counts().sort_index()
after = kept.diagnosis.value_counts().sort_index()
table = pd.DataFrame({
    "clase": [CLASS_NAMES[c] for c in before.index],
    "antes": before.values,
    "después": after.reindex(before.index, fill_value=0).values,
}, index=before.index)
table["quitadas"] = table.antes - table.después
table["% quitado"] = (100 * table.quitadas / table.antes).round(1)
display(table)

x = np.arange(len(table))
fig, ax = plt.subplots(figsize=(8, 3.5))
ax.bar(x - 0.2, table.antes, 0.4, label="antes", color="#9e9e9e")
ax.bar(x + 0.2, table.después, 0.4, label="después", color="#2e75b6")
ax.set_xticks(x, table.clase)
ax.set_ylabel("imágenes"); ax.set_title("Imágenes por clase antes y después de quitar duplicados"); ax.legend()
plt.tight_layout()
plt.show()

## 2. División 70 / 15 / 15

Estratificada por clase: cada parte conserva la proporción de clases del dataset. La semilla fija (`SEED`)
hace que la división salga igual cada vez que se corre.

In [ ]:
splits = split_dataset(kept, seed=SEED)

counts = pd.crosstab(splits.diagnosis.map(CLASS_NAMES), splits.split)[SPLIT_ORDER]
counts = counts.reindex([CLASS_NAMES[c] for c in sorted(CLASS_NAMES)])
counts.loc["Total"] = counts.sum()
counts["total"] = counts.sum(axis=1)
display(counts)

sizes = splits.split.value_counts()[SPLIT_ORDER]
print("Proporción real:", {k: f"{100 * v / len(splits):.1f}%" for k, v in sizes.items()},
      "· pedida:", {k: f"{100 * v:.0f}%" for k, v in SPLITS.items()})

In [ ]:
pct = pd.crosstab(splits.split, splits.diagnosis, normalize="index").loc[SPLIT_ORDER] * 100
pct.loc["dataset"] = kept.diagnosis.value_counts(normalize=True).sort_index() * 100

class_colors = ["#2e7d32", "#7cb342", "#f5a623", "#e53935", "#8e0000"]
fig, ax = plt.subplots(figsize=(8, 3))
left = np.zeros(len(pct))
for c in pct.columns:
    ax.barh(pct.index, pct[c], left=left, color=class_colors[c], label=CLASS_NAMES[c])
    for y, (l, v) in enumerate(zip(left, pct[c])):
        if v > 4:
            ax.text(l + v / 2, y, f"{v:.0f}%", ha="center", va="center", color="white", fontsize=9)
    left += pct[c].to_numpy()
ax.invert_yaxis()
ax.set_xlim(0, 100); ax.set_xlabel("% de imágenes")
ax.set_title("Proporción de clases en cada parte")
ax.legend(ncol=5, loc="upper center", bbox_to_anchor=(0.5, -0.25), fontsize=8, frameon=False)
plt.tight_layout()
plt.show()

## 3. Verificaciones

Si alguna falla, el notebook se detiene aquí y **no** se guarda `splits.csv`.

In [ ]:
checks = {}

# Ninguna imagen en dos partes, y todas las que quedaron están en alguna
checks["cada imagen en una sola parte"] = splits.id_code.is_unique
checks["no falta ni sobra ninguna imagen"] = set(splits.id_code) == set(kept.id_code)

# Ninguna imagen quitada se coló, y no hay dos imágenes del mismo grupo de duplicados
checks["sin imágenes quitadas"] = not set(splits.id_code) & set(removed.id_code)
in_groups = splits.merge(duplicates[["id_code", "dup_group"]], on="id_code")
checks["máximo una imagen por grupo de duplicados"] = in_groups.dup_group.is_unique

# Proporciones y estratificación
real = splits.split.value_counts(normalize=True)
checks["tamaños a ±1% de 70/15/15"] = all(abs(real[k] - v) < 0.01 for k, v in SPLITS.items())
checks["las 5 clases en cada parte"] = all(splits[splits.split == s].diagnosis.nunique() == 5 for s in SPLIT_ORDER)
checks["proporción de clases a ±1.5 puntos del dataset"] = (pct.loc[SPLIT_ORDER] - pct.loc["dataset"]).abs().max().max() < 1.5

# Coincide con train.csv
merged = splits.merge(labels, on="id_code", suffixes=("", "_orig"))
checks["etiquetas iguales a train.csv"] = (merged.diagnosis == merged.diagnosis_orig).all()

for name, ok in checks.items():
    print(("✅" if ok else "❌"), name)
assert all(checks.values()), "Alguna verificación falló: revisar antes de guardar"

## 4. Cámaras en cada parte

El EDA mostró que la resolución (la cámara) se relaciona mucho con la clase. La división es estratificada por clase,
no por cámara, así que revisamos que las cámaras también queden repartidas: si una cámara faltara en prueba, no
podríamos medir el error con esas fotos.

In [ ]:
by_res = splits.merge(stats[["id_code", "resolution"]], on="id_code")
top = by_res.resolution.value_counts().index[:10]
res_pct = pd.crosstab(by_res.resolution, by_res.split, normalize="columns")[SPLIT_ORDER].loc[top] * 100
res_pct["dataset"] = by_res.resolution.value_counts(normalize=True).loc[top] * 100

fig, ax = plt.subplots(figsize=(8, 4.5))
im = ax.imshow(res_pct.to_numpy(), cmap="Blues", aspect="auto")
ax.set_xticks(range(res_pct.shape[1]), res_pct.columns)
ax.set_yticks(range(len(top)), top)
for i in range(res_pct.shape[0]):
    for j in range(res_pct.shape[1]):
        v = res_pct.iat[i, j]
        ax.text(j, i, f"{v:.1f}", ha="center", va="center", fontsize=8, color="white" if v > 15 else "black")
ax.set_title("% de cada parte por resolución (10 más comunes)")
plt.colorbar(im, ax=ax, label="%")
plt.tight_layout()
plt.show()

missing = {s: sorted(set(by_res.resolution) - set(by_res[by_res.split == s].resolution)) for s in SPLIT_ORDER}
print("Resoluciones que no aparecen en cada parte:", missing)

## 5. Guardar

`data/splits.csv` (`id_code,diagnosis,split`) va al repo. También se guarda una copia de `duplicates.csv` en `data/`
para que se pueda volver a generar la división sin correr el EDA. Después: `git add data/splits.csv data/duplicates.csv`.

In [ ]:
OUT.parent.mkdir(parents=True, exist_ok=True)
splits.to_csv(OUT, index=False)
duplicates.to_csv(OUT.parent / "duplicates.csv", index=False)
print(f"Guardado {OUT.relative_to(REPO)} ({len(splits)} filas) y data/duplicates.csv")
splits.head()

## 6. Conclusiones

Llenar después de correr el notebook:

- **Duplicados:** …
- **Tamaños:** train …, val …, test ….
- **Clases y cámaras:** …